# InceptionV3 Training Visualisation
Reads CSV logs from `data/models/{MODEL_NAME}/` and plots a full training dashboard. No running model needed — logs only.

In [1]:
"""
Training Visualisation

Reads CSV logs produced by Keras CSVLogger from:
    data/models/{MODEL_NAME}/

Plots a complete dashboard:
  Row 0 — Top-1 / Top-5 accuracy  |  Loss
  Row 1 — Learning-rate schedule   |  Overfitting gap  |  ΔVal/epoch
  Row 2 — 20 worst classes (recall bar chart)
  Row 3 — 20 best  classes (recall bar chart)

Commented-out blocks at the bottom save every subplot as an SVG.
"""


'\nTraining Visualisation\n\nReads CSV logs produced by Keras CSVLogger from:\n    data/models/{MODEL_NAME}/\n\nPlots a complete dashboard:\n  Row 0 — Top-1 / Top-5 accuracy  |  Loss\n  Row 1 — Learning-rate schedule   |  Overfitting gap  |  ΔVal/epoch\n  Row 2 — 20 worst classes (recall bar chart)\n  Row 3 — 20 best  classes (recall bar chart)\n\nCommented-out blocks at the bottom save every subplot as an SVG.\n'

## 1. Imports

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path


## 2. Configuration
Edit `MODEL_NAME`, `BASE_DIR`, and `STEPS_PER_EPOCH` to match your setup.

In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME  = "InceptionV3"
BASE_DIR    = Path("data/models")
MODEL_DIR   = BASE_DIR / MODEL_NAME

# Log file paths (adjust names if you renamed them)
LOG_BASE  = MODEL_DIR / f"{MODEL_NAME}_history_base.log"
LOG_FT    = MODEL_DIR / f"{MODEL_NAME}_history_finetuned.log"

# Optional: path to per-class classification report CSV
# Expected columns: class, precision, recall, f1-score, support
# If not available, the per-class rows are skipped gracefully.
LOG_REPORT = MODEL_DIR / f"{MODEL_NAME}_class_report.csv"

# Output directory for SVG exports (used in the commented-out section below)
SVG_DIR = MODEL_DIR / "plots"
SVG_DIR.mkdir(parents=True, exist_ok=True)

# Phase 2 cosine-decay parameters — must match your fine-tuning call
FT_INIT_LR     = 1e-5
FT_ALPHA_LR    = 1e-7
STEPS_PER_EPOCH = 4735   # len(train_ds) — update if different


## 3. Load & merge phase logs

In [7]:
# ── Load logs ─────────────────────────────────────────────────────────────────
ht = pd.read_csv(LOG_BASE)
ft = pd.read_csv(LOG_FT)

n_ht = len(ht)
n_ft = len(ft)
epochs_ht  = list(range(1, n_ht + 1))
epochs_ft  = list(range(n_ht + 1, n_ht + n_ft + 1))
epochs_all = epochs_ht + epochs_ft

acc_train  = ht["accuracy"].tolist()      + ft["accuracy"].tolist()
acc_val    = ht["val_accuracy"].tolist()  + ft["val_accuracy"].tolist()
loss_train = ht["loss"].tolist()          + ft["loss"].tolist()
loss_val   = ht["val_loss"].tolist()      + ft["val_loss"].tolist()
top5_train = ht["top5_acc"].tolist()      + ft["top5_acc"].tolist()
top5_val   = ht["val_top5_acc"].tolist()  + ft["val_top5_acc"].tolist()

# Phase 1 LR is logged; Phase 2 uses cosine decay — reconstruct it
lr_ht = ht["learning_rate"].tolist()
t      = np.linspace(0, STEPS_PER_EPOCH * n_ft - 1, n_ft)
lr_ft  = (FT_INIT_LR * 0.5 * (1 + np.cos(np.pi * t / (STEPS_PER_EPOCH * n_ft))) + FT_ALPHA_LR).tolist()
lr_all = lr_ht + lr_ft

print(f"Phase 1 epochs : {n_ht}  |  Phase 2 epochs : {n_ft}")
print(f"Best val Top-1 : {max(acc_val)*100:.2f}%  at epoch {epochs_all[acc_val.index(max(acc_val))]}")
print(f"Final val Top-1: {acc_val[-1]*100:.2f}%   Final val Top-5: {top5_val[-1]*100:.2f}%")


FileNotFoundError: [Errno 2] No such file or directory: 'data/models/InceptionV3/InceptionV3_history_base.log'

## 4. Per-class report *(optional)*
Generate with `pd.DataFrame(classification_report(..., output_dict=True)).T.to_csv(LOG_REPORT)` after evaluation.

In [ ]:
# ── Load per-class report (optional) ─────────────────────────────────────────
# Expected: a CSV with columns [class, precision, recall, f1-score, support] 
# You can generate it after evaluation with:
#   df = pd.DataFrame(classification_report(..., output_dict=True)).T
#   df.to_csv(LOG_REPORT)

has_report = LOG_REPORT.exists()
if has_report:
    rpt = pd.read_csv(LOG_REPORT, index_col=0)
    # Drop summary rows (accuracy / macro avg / weighted avg)
    rpt = rpt[~rpt.index.isin(["accuracy", "macro avg", "weighted avg"])]
    rpt = rpt.dropna(subset=["recall"])
    mean_recall = rpt["recall"].mean()
    worst20 = rpt.nsmallest(20, "recall")
    best20  = rpt.nlargest(20,  "recall")
    print(f"Report loaded — {len(rpt)} classes  |  mean recall {mean_recall:.3f}")
else:
    print("No per-class report found — skipping worst/best class panels.")
    print(f"Expected path: {LOG_REPORT}")


## 5. Theme helpers

In [ ]:
# ── Theme helpers ─────────────────────────────────────────────────────────────
HT_C    = "#61afef"   # blue  — phase 1 (head training)
FT_C    = "#e06c75"   # red   — phase 2 (fine-tuning)
TRAIN_C = "#56b6c2"
VAL_C   = "#e5c07b"
GOOD_C  = "#98c379"
BG      = "#1e2127"
AX_BG   = "#282c34"
TICK_C  = "#abb2bf"

def _style(ax, title):
    ax.set_facecolor(AX_BG)
    ax.set_title(title, color="white", fontsize=10, fontweight="bold", pad=7)
    ax.tick_params(colors=TICK_C)
    ax.xaxis.label.set_color(TICK_C)
    ax.yaxis.label.set_color(TICK_C)
    for sp in ax.spines.values():
        sp.set_edgecolor("#3e4451")
    ax.legend(framealpha=0.3, labelcolor="white", facecolor="#2c313c", fontsize=8)

def _phase_bg(ax):
    ax.axvspan(epochs_ht[0] - 0.5, epochs_ht[-1] + 0.5, alpha=0.06, color=HT_C)
    ax.axvspan(epochs_ft[0] - 0.5, epochs_ft[-1] + 0.5, alpha=0.06, color=FT_C)
    ax.axvline(n_ht + 0.5, color=TICK_C, lw=0.9, ls="--", alpha=0.5)


## 6. Dashboard

In [ ]:
# ── Build dashboard ───────────────────────────────────────────────────────────
n_rows = 4 if has_report else 2
fig = plt.figure(figsize=(20, 6 * n_rows))
fig.patch.set_facecolor(BG)
gs  = GridSpec(n_rows, 3, figure=fig, hspace=0.48, wspace=0.35)

def _ax(r, c, cs=1):
    return fig.add_subplot(gs[r, c:c+cs])

# ── ROW 0 : Accuracy & Loss ───────────────────────────────────────────────────
ax_acc = _ax(0, 0, 2)
ax_acc.plot(epochs_all, acc_train,  color=TRAIN_C, lw=2,   label="Train Top-1")
ax_acc.plot(epochs_all, acc_val,    color=VAL_C,   lw=2,   label="Val   Top-1")
ax_acc.plot(epochs_all, top5_train, color=TRAIN_C, lw=1.2, ls="--", alpha=0.7, label="Train Top-5")
ax_acc.plot(epochs_all, top5_val,   color=VAL_C,   lw=1.2, ls="--", alpha=0.7, label="Val   Top-5")
ax_acc.set_xlabel("Epoch"); ax_acc.set_ylabel("Accuracy")
_style(ax_acc, "Accuracy Over Time  (solid = Top-1  |  dashed = Top-5)")
_phase_bg(ax_acc)

best_ep  = epochs_all[acc_val.index(max(acc_val))]
best_val = max(acc_val)
ax_acc.annotate(
    f"Best val {best_val*100:.1f}% @ ep {best_ep}",
    xy=(best_ep, best_val),
    xytext=(20, -30), textcoords="offset points",
    color="white", fontsize=9,
    arrowprops=dict(arrowstyle="->", color=TICK_C))

ax_loss = _ax(0, 2)
ax_loss.plot(epochs_all, loss_train, color=TRAIN_C, lw=2, label="Train")
ax_loss.plot(epochs_all, loss_val,   color=VAL_C,   lw=2, label="Val")
ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("Loss (label-smoothed CE)")
_style(ax_loss, "Loss Over Time")
_phase_bg(ax_loss)

# ── ROW 1 : LR | Overfitting gap | ΔVal ──────────────────────────────────────
ax_lr = _ax(1, 0)
ax_lr.plot(epochs_all, lr_all, color="#c678dd", lw=2)
ax_lr.set_yscale("log")
ax_lr.set_xlabel("Epoch"); ax_lr.set_ylabel("LR (log scale)")
_style(ax_lr, "Learning Rate Schedule")
_phase_bg(ax_lr)
ax_lr.text(n_ht / 2 + 0.5,         max(lr_all) * 0.6, "Head
Phase",
           color=HT_C, fontsize=8, ha="center")
ax_lr.text(n_ht + n_ft / 2 + 0.5,  max(lr_all) * 0.6, "Fine-tune
Cosine",
           color=FT_C, fontsize=8, ha="center")

gap = [tr - va for tr, va in zip(acc_train, acc_val)]
ax_gap = _ax(1, 1)
ax_gap.bar(epochs_all, gap,
           color=[FT_C if e > n_ht else HT_C for e in epochs_all], alpha=0.8, width=0.8)
ax_gap.axhline(0, color=TICK_C, lw=0.8, ls="--")
ax_gap.set_xlabel("Epoch"); ax_gap.set_ylabel("Train − Val Accuracy")
_style(ax_gap, "Overfitting Gap  (Train − Val Top-1)")
ax_gap.legend(handles=[mpatches.Patch(color=HT_C, label="Phase 1"),
                        mpatches.Patch(color=FT_C, label="Phase 2")],
              framealpha=0.3, labelcolor="white", facecolor="#2c313c", fontsize=8)

deltas = [0] + [acc_val[i] - acc_val[i-1] for i in range(1, len(acc_val))]
ax_delta = _ax(1, 2)
ax_delta.bar(epochs_all, deltas,
             color=[GOOD_C if d >= 0 else FT_C for d in deltas], alpha=0.85, width=0.8)
ax_delta.axhline(0, color=TICK_C, lw=0.8, ls="--")
ax_delta.set_xlabel("Epoch"); ax_delta.set_ylabel("ΔVal Top-1")
_style(ax_delta, "Val Accuracy Δ per Epoch")

# ── ROW 2 & 3 : Per-class performance (only if report exists) ─────────────────
if has_report:
    ax_worst = _ax(2, 0, 3)
    yp = np.arange(len(worst20))
    bars_w = ax_worst.barh(yp, worst20["recall"].values, color=FT_C, alpha=0.85)
    ax_worst.set_yticks(yp)
    ax_worst.set_yticklabels(worst20.index.tolist(), fontsize=9, color="white")
    ax_worst.set_xlabel("Recall")
    ax_worst.axvline(mean_recall, color=VAL_C, ls="--", lw=1.3,
                     label=f"Mean {mean_recall:.2f}")
    for bar, v in zip(bars_w, worst20["recall"].values):
        ax_worst.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height() / 2,
                      f"{v:.2f}", va="center", color="white", fontsize=8)
    _style(ax_worst, "20 Worst-Performing Classes — Recall")

    ax_best = _ax(3, 0, 3)
    yp2 = np.arange(len(best20))
    bars_b = ax_best.barh(yp2, best20["recall"].values, color=GOOD_C, alpha=0.85)
    ax_best.set_yticks(yp2)
    ax_best.set_yticklabels(best20.index.tolist(), fontsize=9, color="white")
    ax_best.set_xlabel("Recall")
    ax_best.axvline(mean_recall, color=VAL_C, ls="--", lw=1.3,
                    label=f"Mean {mean_recall:.2f}")
    for bar, v in zip(bars_b, best20["recall"].values):
        ax_best.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height() / 2,
                     f"{v:.2f}", va="center", color="white", fontsize=8)
    _style(ax_best, "20 Best-Performing Classes — Recall")

# ── Suptitle ──────────────────────────────────────────────────────────────────
fig.suptitle(
    f"{MODEL_NAME} — Training Dashboard  "
    f"│  Best Val Top-1 {max(acc_val)*100:.2f}%  "
    f"│  Final Val Top-5 {top5_val[-1]*100:.2f}%  "
    f"│  Phase 1: {n_ht} ep  │  Phase 2: {n_ft} ep",
    color="white", fontsize=13, fontweight="bold", y=0.999)

plt.savefig(MODEL_DIR / "training_dashboard.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Dashboard PNG saved → {MODEL_DIR / 'training_dashboard.png'}")


## 7. SVG export
Uncomment the block below to save each subplot as an individual vector SVG into `SVG_DIR`.

In [ ]:
# ── SVG export — uncomment to save individual subplots ───────────────────────
# Each subplot is re-drawn into its own figure and saved as a vector SVG.
#
# def _save_svg(draw_fn, filename, figsize=(10, 5)):
#     fig_s, ax_s = plt.subplots(figsize=figsize)
#     fig_s.patch.set_facecolor(BG)
#     draw_fn(ax_s)
#     fig_s.savefig(SVG_DIR / filename, format="svg",
#                   bbox_inches="tight", facecolor=fig_s.get_facecolor())
#     plt.close(fig_s)
#     print(f"Saved {SVG_DIR / filename}")
#
# # --- accuracy_top1_top5.svg ---
# def _draw_acc(ax):
#     ax.plot(epochs_all, acc_train,  color=TRAIN_C, lw=2,   label="Train Top-1")
#     ax.plot(epochs_all, acc_val,    color=VAL_C,   lw=2,   label="Val   Top-1")
#     ax.plot(epochs_all, top5_train, color=TRAIN_C, lw=1.2, ls="--", alpha=0.7, label="Train Top-5")
#     ax.plot(epochs_all, top5_val,   color=VAL_C,   lw=1.2, ls="--", alpha=0.7, label="Val   Top-5")
#     ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
#     _phase_bg(ax); _style(ax, "Accuracy Over Time")
# _save_svg(_draw_acc, "accuracy_top1_top5.svg")
#
# # --- loss.svg ---
# def _draw_loss(ax):
#     ax.plot(epochs_all, loss_train, color=TRAIN_C, lw=2, label="Train")
#     ax.plot(epochs_all, loss_val,   color=VAL_C,   lw=2, label="Val")
#     ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
#     _phase_bg(ax); _style(ax, "Loss Over Time")
# _save_svg(_draw_loss, "loss.svg")
#
# # --- learning_rate.svg ---
# def _draw_lr(ax):
#     ax.plot(epochs_all, lr_all, color="#c678dd", lw=2)
#     ax.set_yscale("log"); ax.set_xlabel("Epoch"); ax.set_ylabel("LR (log)")
#     _phase_bg(ax); _style(ax, "Learning Rate Schedule")
# _save_svg(_draw_lr, "learning_rate.svg")
#
# # --- overfitting_gap.svg ---
# def _draw_gap(ax):
#     ax.bar(epochs_all, gap, color=[FT_C if e > n_ht else HT_C for e in epochs_all],
#            alpha=0.8, width=0.8)
#     ax.axhline(0, color=TICK_C, lw=0.8, ls="--")
#     ax.set_xlabel("Epoch"); ax.set_ylabel("Train − Val Accuracy")
#     _style(ax, "Overfitting Gap")
# _save_svg(_draw_gap, "overfitting_gap.svg")
#
# # --- val_delta.svg ---
# def _draw_delta(ax):
#     ax.bar(epochs_all, deltas, color=[GOOD_C if d >= 0 else FT_C for d in deltas],
#            alpha=0.85, width=0.8)
#     ax.axhline(0, color=TICK_C, lw=0.8, ls="--")
#     ax.set_xlabel("Epoch"); ax.set_ylabel("ΔVal Top-1")
#     _style(ax, "Val Accuracy Δ per Epoch")
# _save_svg(_draw_delta, "val_delta.svg")
#
# # --- worst20_recall.svg (requires report) ---
# if has_report:
#     def _draw_worst(ax):
#         yp = np.arange(len(worst20))
#         ax.barh(yp, worst20["recall"].values, color=FT_C, alpha=0.85)
#         ax.set_yticks(yp); ax.set_yticklabels(worst20.index.tolist(), fontsize=9, color="white")
#         ax.axvline(mean_recall, color=VAL_C, ls="--", lw=1.3, label=f"Mean {mean_recall:.2f}")
#         ax.set_xlabel("Recall"); _style(ax, "20 Worst Classes — Recall")
#     _save_svg(_draw_worst, "worst20_recall.svg", figsize=(12, 7))
#
#     def _draw_best(ax):
#         yp = np.arange(len(best20))
#         ax.barh(yp, best20["recall"].values, color=GOOD_C, alpha=0.85)
#         ax.set_yticks(yp); ax.set_yticklabels(best20.index.tolist(), fontsize=9, color="white")
#         ax.axvline(mean_recall, color=VAL_C, ls="--", lw=1.3, label=f"Mean {mean_recall:.2f}")
#         ax.set_xlabel("Recall"); _style(ax, "20 Best Classes — Recall")
#     _save_svg(_draw_best, "best20_recall.svg", figsize=(12, 7))
#
# print(f"All SVGs saved to {SVG_DIR}")
